In [1]:
# Verify that CUDA is available and that Google Colab assigned a GPU.
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [2]:
# Install InsightFace and the GPU version of ONNX Runtime
!pip install insightface onnxruntime-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 762.2/762.2 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 67.5 MB/s eta 0:00:00


In [3]:
import cv2
import matplotlib.pyplot as plt
from insightface.app import FaceAnalysis

# Initialize FaceAnalysis and force it to use the GPU (ctx_id=0 targets the first GPU)
app = FaceAnalysis(providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

print("InsightFace successfully initialized on GPU!")

download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:07<00:00, 38205.94KB/s]
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:149: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)
InsightFace 

In [4]:
import cv2
import os
from tqdm import tqdm  # Visual progress bar

def process_and_save_dataset(input_dir, output_dir, insightface_app):
    """
    Scans the input directory, detects faces, applies a 15% padding,
    resizes them to 224x224, and saves them to the output directory.
    """
    # Create target directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Filter image files
    valid_extensions = ('.png', '.jpg', '.jpeg')
    files = [f for f in os.listdir(input_dir) if f.lower().endswith(valid_extensions)]
    print(f"Found {len(files)} images to process in '{input_dir}'")

    # Added tqdm here for a clean progress bar instead of infinite text flooding
    for filename in tqdm(files, desc="Processing images via GPU"):
        img_path = os.path.join(input_dir, filename)
        img = cv2.imread(img_path)

        if img is None:
            continue

        # 1. Face Detection
        faces = insightface_app.get(img)
        if not faces:
            continue

        # Extract the primary face
        face = faces[0]

        # 2. Strict Preprocessing Logic (15% Padding)
        x1, y1, x2, y2 = face.bbox.astype(int)
        h, w = img.shape[:2]

        face_width = x2 - x1
        face_height = y2 - y1

        padding_x = int(face_width * 0.15)
        padding_y = int(face_height * 0.15)

        x1 = max(0, x1 - padding_x)
        y1 = max(0, y1 - padding_y)
        x2 = min(w, x2 + padding_x)
        y2 = min(h, y2 + padding_y)

        face_img = img[y1:y2, x1:x2]
        face_img = cv2.resize(face_img, (224, 224))

        # 3. Save the processed 224x224 crop
        output_path = os.path.join(output_dir, filename)
        cv2.imwrite(output_path, face_img)

    print("\nProcessing complete!")

In [ ]:
from google.colab import files
uploaded = files.upload()
process_and_save_dataset(input_dir=".", output_dir="processed_faces", insightface_app=app)

Saving 859A9168.JPG to 859A9168.JPG
Found 1 images to process in '.'


Processing images via GPU: 100%|██████████| 1/1 [00:01<00:00,  1.52s/it]


Processing complete!


In [ ]:
import os
import json

# 1. Input your Kaggle credentials directly
KAGGLE_USERNAME = input("Enter your Kaggle Username: ").strip()
KAGGLE_KEY = input("Enter your Kaggle API Key: ").strip()

# 2. Automatically create the structure expected by the system
kaggle_data = {"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump(kaggle_data, f)

# Set secure permissions
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("\nKaggle credentials configured successfully from API Key!")

# 3. Download the dataset directly
print("\nDownloading CelebA dataset...")
!kaggle datasets download -d jessicali9530/celeba-dataset

# 4. Extract files
print("\nExtracting dataset... (this takes about a minute)")
!mkdir -p celeba_dataset
!unzip -q celeba-dataset.zip -d celeba_dataset

# 5. Clean up zip
!rm celeba-dataset.zip
print("\nDownload and extraction complete!")

# 6. Verify files
img_dir = "celeba_dataset/img_align_celeba/img_align_celeba"
if os.path.exists(img_dir):
    print(f"SUCCESS: Found {len(os.listdir(img_dir))} images in the directory.")

Enter your Kaggle Username: avigailstachi
Enter your Kaggle API Key: KGAT_62a4e901d2f770b84d9b6543b7dddc39

Kaggle credentials configured successfully from API Key!

Dataset URL: https://www.kaggle.com/datasets/jessicali9530/celeba-dataset
License(s): other
100% 1.33G/1.33G [00:07<00:00, 201MB/s]


Extracting dataset... (this takes about a minute)

Download and extraction complete!
SUCCESS: Found 202599 images in the directory.


In [ ]:
process_and_save_dataset(
    input_dir="celeba_dataset/img_align_celeba/img_align_celeba",
    output_dir="processed_celeba_faces",
    insightface_app=app
)

Found 202599 images to process in 'celeba_dataset/img_align_celeba/img_align_celeba'


Processing images via GPU:   2%|▏         | 3959/202599 [51:16<43:52:56,  1.26it/s]

In [ ]:
import pandas as pd

# Load attributes (Kaggle extracted it as a CSV file)
attr_df = pd.read_csv("celeba_dataset/list_attr_celeba.csv")

# Keep only Image name and the 'Smiling' attribute
smiling_df = attr_df[['image_id', 'Smiling']].copy()

# Convert Kaggle's (-1) labels to (0) for standard binary classification
smiling_df['Smiling'] = smiling_df['Smiling'].replace(-1, 0)

print(smiling_df.head())

In [ ]:
# Get list of successfully processed images on your disk
processed_images = set(os.listdir("processed_celeba_faces"))

# Filter the dataframe to match only existing files
final_dataset_df = smiling_df[smiling_df['image_id'].isin(processed_images)].reset_index(drop=True)

print(f"Total labeled images ready for training: {len(final_dataset_df)}")

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

def get_smile_model():
    # Load pre-trained MobileNetV3 Large
    weights = MobileNet_V3_Large_Weights.DEFAULT
    model = mobilenet_v3_large(weights=weights)

    # Modify the final classifier layer for binary classification
    # MobileNetV3's classifier is a Sequential block; the last layer is a Linear layer
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_features, 1)

    return model

# Initialize and move to your T4 GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = get_smile_model().to(device)

In [ ]:
# Standard binary classification setup
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# Example dummy forward pass to test architecture
dummy_input = torch.randn(16, 3, 224, 224).to(device) # Batch of 16 images
logits = model(dummy_input) # Outputs shape: [16, 1]

# Example loss calculation
dummy_labels = torch.empty(16, 1).random_(2).to(device) # 0s and 1s
loss = criterion(logits, dummy_labels)
print(f"Initial loss: {loss.item()}")

In [ ]:
# This will print the exact content of your current Colab notebook
!cat /content/*.ipynb 2>/dev/null || cat *.ipynb